
# پروژه کلاس بندی ژانر موسیقی 
برای رسیدن به این هدف نیاز داریم که داده هایی زیادی در ژانر های متفاوت داشته باشیم دیتاست 

gtzan 

در ده کلاس (ژانر) و هر کدام از کلاس ها ۱۰۰ نمونه موسیقی به مدت ۳۰ ثانیه تهیه کرده است

## تقسیم هر موسیقی به بخش های کوچکتر

هر ترک این دیتاست در قالب موسیقی ۳۰ ثانیه وجود دارد و باید این ۳۰ ثانیه را به بخش های کوچکتری تقسیم کنیم تا بتوانیم الگو را در هر قسمت پیدا کنیم که به این فرآیند

Framing 

گفته می شود

برای اینکه بدانیم هر موسیقی را به چند قسمت چند ثانیه ای تقسیم کنیم نیاز به تکرار و آزمایش کردن داریم

زمانی هم که هر موسیقی را به قسمت های کوچکتر تقسیم می کنیم باید کاری کنیم که از 

data leakage 

جلوگیری شود 

برای این موضوع نیز باید هر قسمت موسیقی که تقسیم می شود همه قسمت های آن در بخش آموزش باید باشد یا حتما همشون در بخش آزمایش


در قسمت 


`make_features.py`

من این فیچرها را ساختم و از این برای مدل هم استفاده می کنم

## ویژگی هایی صدا
- MFCC (mel frequency cepstral coefficients)
- Spectral Centroid
- Band Energy Ratio
- BandWidth


### وارد کردن کتابخانه 

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone

from sklearn.metrics import average_precision_score, f1_score

RANDOM_STATE = 42


In [4]:
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

In [5]:
df = pd.read_csv('/Users/pouniq/music_genre_classification/audio_features/gtzan_5s_features.csv')

In [6]:
mfcc_features = [
    "mfcc1_mean", "mfcc1_var",
    "mfcc2_mean", "mfcc2_var",
    "mfcc3_mean", "mfcc3_var",
    "mfcc4_mean", "mfcc4_var",
    "mfcc5_mean", "mfcc5_var",
    "mfcc6_mean", "mfcc6_var",
    "mfcc7_mean", "mfcc7_var",
    "mfcc8_mean", "mfcc8_var",
    "mfcc9_mean", "mfcc9_var",
    "mfcc10_mean", "mfcc10_var",
    "mfcc11_mean", "mfcc11_var",
    "mfcc12_mean", "mfcc12_var",
    "mfcc13_mean", "mfcc13_var",
    "mfcc14_mean", "mfcc14_var",
    "mfcc15_mean", "mfcc15_var",
    "mfcc16_mean", "mfcc16_var",
    "mfcc17_mean", "mfcc17_var",
    "mfcc18_mean", "mfcc18_var",
    "mfcc19_mean", "mfcc19_var",
    "mfcc20_mean", "mfcc20_var",
    "ber1_mean","ber1_var"
]

In [7]:
X = df.drop(columns=["file", "genre", "segment", "start_sec"])
y = df["genre"]
groups = df["file"]

In [8]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
model = make_pipeline(StandardScaler(), RandomForestClassifier())

scores = cross_validate(model, X, y, groups=groups, cv=cv,
                        scoring=["accuracy", "f1_macro"])

In [9]:
print(scores["test_f1_macro"].mean(), scores["test_f1_macro"].std())


0.7092137337297799 0.023744856064133908


In [10]:
for tr, te in cv.split(X, y, groups):
    assert set(groups.iloc[tr]).isdisjoint(groups.iloc[te])

In [28]:
groups.nunique()

999

In [13]:
print(X.columns.tolist())

['chroma1_mean', 'chroma1_var', 'chroma2_mean', 'chroma2_var', 'chroma3_mean', 'chroma3_var', 'chroma4_mean', 'chroma4_var', 'chroma5_mean', 'chroma5_var', 'chroma6_mean', 'chroma6_var', 'chroma7_mean', 'chroma7_var', 'chroma8_mean', 'chroma8_var', 'chroma9_mean', 'chroma9_var', 'chroma10_mean', 'chroma10_var', 'chroma11_mean', 'chroma11_var', 'chroma12_mean', 'chroma12_var', 'rms1_mean', 'rms1_var', 'spectral_centroid1_mean', 'spectral_centroid1_var', 'spectral_bandwidth1_mean', 'spectral_bandwidth1_var', 'rolloff1_mean', 'rolloff1_var', 'zcr1_mean', 'zcr1_var', 'contrast1_mean', 'contrast1_var', 'contrast2_mean', 'contrast2_var', 'contrast3_mean', 'contrast3_var', 'contrast4_mean', 'contrast4_var', 'contrast5_mean', 'contrast5_var', 'contrast6_mean', 'contrast6_var', 'contrast7_mean', 'contrast7_var', 'mfcc1_mean', 'mfcc1_var', 'mfcc2_mean', 'mfcc2_var', 'mfcc3_mean', 'mfcc3_var', 'mfcc4_mean', 'mfcc4_var', 'mfcc5_mean', 'mfcc5_var', 'mfcc6_mean', 'mfcc6_var', 'mfcc7_mean', 'mfcc7_va

## MODEL SELECTION
- SVC
- RF
- LOGISTIC REGRESSION
- NAIVE BAYES


In [15]:
models = {
    'svc': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(random_state=RANDOM_STATE)),
    ]),
    'hgb': Pipeline([
        ('model', HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
    'rf': Pipeline([
        ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    'lg': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    
    'lda': Pipeline(
        [
            ('scaler', StandardScaler()),
            ('model', LinearDiscriminantAnalysis())
        ]
    )
}

In [16]:
y = y.astype('category').cat.codes
y

0       0
1       0
2       0
3       0
4       0
       ..
5980    9
5981    9
5982    9
5983    9
5984    9
Length: 5985, dtype: int8

In [17]:
for name, m in models.items():

    f1 = []

    for train_idx, test_idx in cv.split(X, y, groups=groups):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        m.fit(X_train, y_train)
        y_pred = m.predict(X_test)

        score = f1_score(
            y_test,
            y_pred=y_pred,
            average='macro'
        )

        f1.append(score)

    print(f"{name}: {np.mean(f1):.4f} (+/- {np.std(f1):.4f})")

svc: 0.7363 (+/- 0.0192)
hgb: 0.7466 (+/- 0.0229)
rf: 0.7032 (+/- 0.0277)
lg: 0.6846 (+/- 0.0109)
lda: 0.6787 (+/- 0.0151)


In [18]:
import joblib

best_model = clone(models['hgb']).fit(X, y)
joblib.dump(best_model, "saved_models/best_model.joblib")



['saved_models/best_model.joblib']

## Hyper-Parameter Tunning for HGB

In [19]:
from scipy.stats import randint, loguniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_dist = {
    'model__learning_rate': loguniform(0.01, 0.3),
    'model__max_iter': randint(100, 501),
    'model__max_leaf_nodes': [15, 31, 63, 127],
    'model__max_depth': [None, 3, 5, 7, 10],
    'model__min_samples_leaf': randint(10, 51),
    'model__l2_regularization': loguniform(1e-4, 10),
    'model__max_bins': [64, 128, 200, 255],
}

In [27]:
hgb_search = RandomizedSearchCV(
    estimator=models['hgb'],
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_macro',
    cv = cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    
)
hgb_search.fit(X_train, y_train,groups=groups)

ValueError: Found input variables with inconsistent numbers of samples: [4791, 4791, 5985]

In [103]:
genre_cat = df["genre"].astype("category")
genre_names = list(genre_cat.cat.categories)

joblib.dump(genre_names, "saved_models/genre_names.joblib")

['saved_models/genre_names.joblib']